# FotMob Player Stats Scraper
**Works for any league and season available on FotMob.**

| Step | Cell | Description |
|------|------|-------------|
| Setup | 1 | Imports, config, constants |
| Auth | 2 | HMAC header + async HTTP client |
| Fixtures | 3 | Fetch all matchday fixtures |
| Scrape | 4–5 | Fetch + parse match pages |
| Run | 6 | Full season scrape |
| Inspect | 7 | Explore metric catalogue |
| Export | 8–10 | Wide-format CSVs + Parquet |

## Cell 1 — Imports & configuration

In [135]:
import hashlib
import hmac
import time
import random
import json
import asyncio
import logging
import requests
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('fotmob')

# ── Change these to scrape a different league or season ───────────
LEAGUE_ID   = 87            # 87=La Liga (PRIMARY TARGET), 47=Premier League, 54=Bundesliga
LEAGUE_SLUG = 'laliga'      # URL slug: 'laliga', 'premier-league', 'bundesliga'
SEASON      = '2021/2022'   # format: 'YYYY/YYYY'  — run 4x: 2021/2022, 2022/2023, 2023/2024, 2024/2025

MAX_CONCURRENT = 4
JITTER_MIN     = 1.0
JITTER_MAX     = 3.0
MAX_RETRIES    = 5
BACKOFF_BASE   = 2

# Storage — organised by league and season so multiple leagues never clash
BASE_DIR = Path('data') / str(LEAGUE_ID) / SEASON.replace('/', '_')
RAW_DIR  = BASE_DIR / 'raw'
OUT_DIR  = BASE_DIR / 'output'
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# FotMob stat category title -> clean snake_case name
CATEGORY_MAP = {
    'Top stats'       : 'top_stats',
    'Attack'          : 'attack',
    'Defence'         : 'defense',
    'Defense'         : 'defense',
    'Passes'          : 'passes',
    'Duels'           : 'duels',
    'Goalkeeping'     : 'goalkeeping',
    'Physical metrics': 'physical_metrics',
}

# Goalkeeper metric keys used to reclassify their single Top stats block
GK_GOALKEEPING = {
    'saves', 'goals_conceded', 'xgot_faced', 'goals_prevented',
    'diving_save', 'saves_inside_box', 'acted_as_sweeper',
    'punches', 'throws', 'high_claim',
}
GK_PASSING = {'accurate_passes', 'accurate_long_balls', 'passes_into_final_third', 'accurate_crosses'}
GK_DEFENSE = {'recoveries', 'clearances', 'defensive_actions', 'tackles', 'interceptions', 'shot_blocks'}
GK_DUELS   = {'ground_duels_won', 'aerials_won', 'was_fouled', 'fouls_committed', 'aerial_duels_won'}

# Metrics excluded from the wide-format export
EXCLUDE_FROM_WIDE = {
    'physical_metrics_distance_covered', 'physical_metrics_topspeed',
    'physical_metrics_running', 'physical_metrics_sprinting',
    'physical_metrics_walking', 'physical_metrics_number_of_sprints',
    'fantasy_points', 'Shotmap',
}

print(f'Config set  --  League {LEAGUE_ID} ({LEAGUE_SLUG})  |  Season {SEASON}')
print(f'  Raw dir : {RAW_DIR.resolve()}')
print(f'  Out dir : {OUT_DIR.resolve()}')


Config set  --  League 87 (laliga)  |  Season 2021/2022
  Raw dir : C:\Users\amine\OneDrive\Documents\git_workspace\GOALS\data\87\2021_2022\raw
  Out dir : C:\Users\amine\OneDrive\Documents\git_workspace\GOALS\data\87\2021_2022\output


## Cell 2 — Auth headers & async HTTP client

In [136]:
# NOTE: The async FotMobClient is no longer needed — FotMob moved fixtures to
# __NEXT_DATA__ HTML embedding (the old /api/leagues endpoint was deprecated).
# Match page scraping still uses synchronous requests (unchanged).

PAGE_HEADERS = {
    'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept'         : 'text/html,application/xhtml+xml,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate',
    'Referer'        : 'https://www.fotmob.com/',
}

print('PAGE_HEADERS defined')


PAGE_HEADERS defined


## Cell 3 — Fetch all fixtures

In [137]:
def fetch_all_fixtures() -> pd.DataFrame:
    """
    Fetch all fixtures for the configured league + season.
    Scrapes the FotMob league fixtures page (__NEXT_DATA__) instead of the
    deprecated /api/leagues endpoint.
    """
    url = (
        f'https://www.fotmob.com/leagues/{LEAGUE_ID}/fixtures/{LEAGUE_SLUG}'
        f'?season={SEASON}'
    )
    resp = requests.get(url, headers=PAGE_HEADERS, timeout=30)
    resp.raise_for_status()

    tag = BeautifulSoup(resp.text, 'html.parser').find('script', {'id': '__NEXT_DATA__'})
    if not tag:
        raise RuntimeError(f'No __NEXT_DATA__ found at {url}')

    data        = json.loads(tag.string)
    page_props  = data.get('props', {}).get('pageProps', {})
    all_matches = page_props.get('fixtures', {}).get('allMatches', [])

    # Save raw league meta for reference
    with open(RAW_DIR / 'league_meta.json', 'w') as f:
        json.dump(page_props, f, indent=2)

    if not all_matches:
        print('No matches found -- check league_meta.json')
        return pd.DataFrame()

    rows = []
    for m in all_matches:
        rows.append({
            'match_id'  : m.get('id'),
            'round'     : m.get('round'),
            'page_url'  : m.get('pageUrl', ''),
            'match_date': m.get('status', {}).get('utcTime', ''),
            'finished'  : m.get('status', {}).get('finished', False),
            'home_team' : m.get('home', {}).get('name', ''),
            'home_id'   : m.get('home', {}).get('id'),
            'away_team' : m.get('away', {}).get('name', ''),
            'away_id'   : m.get('away', {}).get('id'),
        })

    df = pd.DataFrame(rows).dropna(subset=['match_id'])
    df['match_id'] = df['match_id'].astype(int)
    df = df.sort_values(['round', 'match_date']).reset_index(drop=True)
    df.to_parquet(OUT_DIR / 'fixtures.parquet', index=False)
    df.to_csv(OUT_DIR / 'fixtures.csv', index=False)
    log.info('Saved %d fixtures across %d rounds', len(df), df['round'].nunique())
    return df


fixtures_df = fetch_all_fixtures()

print(f'Total    : {len(fixtures_df)}')
print(f'Rounds   : {fixtures_df["round"].nunique()}')
print(f'Finished : {fixtures_df["finished"].sum()}')
print(f'Unplayed : {(~fixtures_df["finished"]).sum()}')
fixtures_df.head()


22:22:24  INFO      Saved 380 fixtures across 38 rounds


Total    : 380
Rounds   : 38
Finished : 380
Unplayed : 0


,match_id,round,page_url,match_date,finished,home_team,home_id,away_team,away_id
0,3629099,1,/matches/getafe-vs-valencia/2uos8v#3629099,2021-08-13T19:00:00Z,True,Valencia,10267,Getafe,8305
1,3629096,1,/matches/cadiz-vs-levante/2dp4je#3629096,2021-08-14T17:30:00Z,True,Cadiz,8385,Levante,8581
2,3629095,1,/matches/real-betis-vs-mallorca/2gqg3x#3629095,2021-08-14T17:30:00Z,True,Mallorca,8661,Real Betis,8603
3,3629092,1,/matches/real-madrid-vs-deportivo-alaves/2tvtd...,2021-08-14T20:00:00Z,True,Deportivo Alaves,9866,Real Madrid,8633
4,3629097,1,/matches/osasuna-vs-espanyol/2dbonz#3629097,2021-08-14T20:00:00Z,True,Osasuna,8371,Espanyol,8558


## Cell 4 — Match page fetcher & parser

In [138]:
def fetch_match_page(page_url: str) -> dict | None:
    """Fetch a match page and extract the embedded __NEXT_DATA__ JSON."""
    full_url = f'https://www.fotmob.com{page_url}'
    try:
        resp = requests.get(full_url, headers=PAGE_HEADERS, timeout=20)
        if resp.status_code != 200:
            log.warning('HTTP %s for %s', resp.status_code, full_url)
            return None
        tag = BeautifulSoup(resp.text, 'html.parser').find('script', {'id': '__NEXT_DATA__'})
        if not tag:
            log.warning('No __NEXT_DATA__ at %s', full_url)
            return None
        return json.loads(tag.string)
    except Exception as exc:
        log.warning('Error fetching %s: %s', full_url, exc)
        return None


def parse_match_page(next_data: dict, match_row: pd.Series) -> list[dict]:
    """
    Parse a match __NEXT_DATA__ blob into flat stat rows.
    Returns one row per (player x match x category x metric).

    Stat block structure per player:
      stats: [ { title, key, stats: { 'Metric Name': { key, stat: {value, type} } } } ]

    Goalkeeper edge case:
      FotMob lumps all GK stats into one Top stats block.
      We reclassify each metric into the correct category
      using the GK_* key sets defined in Cell 1.
    """
    rows         = []
    page_props   = next_data.get('props', {}).get('pageProps', {})
    content      = page_props.get('content', {})
    player_stats = content.get('playerStats', {})

    if not player_stats:
        return rows

    match_ctx = {
        'match_id'  : match_row['match_id'],
        'round'     : match_row['round'],
        'match_date': match_row['match_date'],
        'home_team' : match_row['home_team'],
        'away_team' : match_row['away_team'],
    }

    for pid_str, pdata in player_stats.items():
        if not isinstance(pdata, dict):
            continue

        is_gk = pdata.get('isGoalkeeper', False)

        player_ctx = {
            'player_id'    : pdata.get('id', pid_str),
            'player_name'  : pdata.get('name', ''),
            'team_id'      : pdata.get('teamId'),
            'team_name'    : pdata.get('teamName', ''),
            'is_goalkeeper': is_gk,
            'shirt_number' : pdata.get('shirtNumber'),
            'position_id'  : pdata.get('usualPosition') or pdata.get('positionId'),
        }

        for block in pdata.get('stats', []):
            if not isinstance(block, dict):
                continue

            raw_title  = block.get('title', '')
            block_key  = block.get('key', '')
            stats_dict = block.get('stats', {})

            if not isinstance(stats_dict, dict):
                continue

            for metric_name, metric_data in stats_dict.items():
                if not isinstance(metric_data, dict):
                    continue

                metric_key = metric_data.get('key', '')
                stat       = metric_data.get('stat', {})
                value      = stat.get('value')
                total      = stat.get('total')
                stat_type  = stat.get('type', '')

                # Reclassify GK metrics out of their single Top stats block
                if is_gk and block_key == 'top_stats':
                    if metric_key in GK_GOALKEEPING:
                        category = 'goalkeeping'
                    elif metric_key in GK_PASSING:
                        category = 'passes'
                    elif metric_key in GK_DEFENSE:
                        category = 'defense'
                    elif metric_key in GK_DUELS:
                        category = 'duels'
                    else:
                        category = 'top_stats'
                else:
                    category = CATEGORY_MAP.get(raw_title, block_key)

                rows.append({
                    **match_ctx,
                    **player_ctx,
                    'category'  : category,
                    'metric'    : metric_name,
                    'metric_key': metric_key,
                    'value'     : value,
                    'total'     : total,
                    'stat_type' : stat_type,
                })

    return rows


print('✓ Fetcher + parser defined')

✓ Fetcher + parser defined


## Cell 5 — Full season scrape

In [139]:
def scrape_full_season(fixtures_df: pd.DataFrame) -> pd.DataFrame:
    """
    Scrape and parse all finished matches.
    - Loads from RAW_DIR/<match_id>.json if already cached (idempotent)
    - Silently skips matches where playerStats is None (unplayed)
    - Re-run at any time to pick up newly finished matches
    """
    finished = fixtures_df[fixtures_df['finished']].copy().reset_index(drop=True)
    all_rows = []
    failed   = []

    already_cached = sum(
        1 for _, r in finished.iterrows()
        if (RAW_DIR / (str(r['match_id']) + '.json')).exists()
    )
    print(f'Finished matches  : {len(finished)}')
    print(f'Already cached    : {already_cached}')
    print(f'New requests      : {len(finished) - already_cached}')
    print()

    for _, match_row in tqdm(finished.iterrows(), total=len(finished), desc='Scraping', unit='match'):
        match_id = match_row['match_id']
        raw_path = RAW_DIR / f'{match_id}.json'

        if raw_path.exists():
            with open(raw_path) as f:
                next_data = json.load(f)
        else:
            time.sleep(random.uniform(JITTER_MIN, JITTER_MAX))
            next_data = fetch_match_page(match_row['page_url'])
            if next_data is None:
                failed.append(match_id)
                continue
            with open(raw_path, 'w') as f:
                json.dump(next_data, f)

        content      = next_data.get('props', {}).get('pageProps', {}).get('content', {})
        player_stats = content.get('playerStats')
        if not player_stats:
            continue

        rows = parse_match_page(next_data, match_row)
        if rows:
            all_rows.extend(rows)
        else:
            failed.append(match_id)

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df['match_id']   = df['match_id'].astype(int)
        df['round']      = df['round'].astype(int)
        df['value']      = pd.to_numeric(df['value'], errors='coerce')
        df['total']      = pd.to_numeric(df['total'], errors='coerce')
        df['match_date'] = pd.to_datetime(df['match_date'], utc=True, errors='coerce')

    print(f'\n{"="*55}')
    print(f'  Total stat rows  : {len(df):,}')
    print(f'  Matches parsed   : {df["match_id"].nunique() if not df.empty else 0}')
    print(f'  Unique players   : {df["player_id"].nunique() if not df.empty else 0}')
    print(f'  Categories       : {sorted(df["category"].unique().tolist()) if not df.empty else []}')
    print(f'  Failed           : {len(failed)}')
    print(f'{"="*55}')
    return df


stats_df = scrape_full_season(fixtures_df)

# Save raw long-format as the master archive
stats_df.to_parquet(OUT_DIR / 'player_stats.parquet', index=False, engine='pyarrow')
stats_df.to_csv(OUT_DIR / 'player_stats.csv', index=False)
print(f'\n✓ player_stats.parquet saved  ({(OUT_DIR / "player_stats.parquet").stat().st_size / 1e6:.1f} MB)')

Finished matches  : 380
Already cached    : 380
New requests      : 0



Scraping: 100%|██████████| 380/380 [00:15<00:00, 25.20match/s]



  Total stat rows  : 222,950
  Matches parsed   : 230
  Unique players   : 688
  Categories       : ['attack', 'defense', 'duels', 'goalkeeping', 'passes', 'top_stats']
  Failed           : 0

✓ player_stats.parquet saved  (0.6 MB)


## Cell 6 — Reparse from cache (re-entry point)
Use this cell when you update the parser and want to reprocess
without re-scraping. Uncomment the last line to reload from saved Parquet.

In [140]:
def reparse_from_cache(fixtures_df: pd.DataFrame) -> pd.DataFrame:
    finished = fixtures_df[fixtures_df['finished']].copy().reset_index(drop=True)
    cached   = [row for _, row in finished.iterrows()
                if (RAW_DIR / f"{row['match_id']}.json").exists()]
    all_rows = []
    failed   = []

    print(f'Cached files to reparse: {len(cached)}')

    for match_row in tqdm(cached, desc='Reparsing', unit='match'):
        match_id = match_row['match_id']
        try:
            with open(RAW_DIR / f'{match_id}.json') as f:
                next_data = json.load(f)
            content      = next_data.get('props', {}).get('pageProps', {}).get('content', {})
            player_stats = content.get('playerStats')
            if not player_stats:
                continue
            rows = parse_match_page(next_data, match_row)
            if rows:
                all_rows.extend(rows)
            else:
                failed.append(match_id)
        except Exception as exc:
            log.warning('Failed reparsing %s: %s', match_id, exc)
            failed.append(match_id)

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df['match_id']   = df['match_id'].astype(int)
        df['round']      = df['round'].astype(int)
        df['value']      = pd.to_numeric(df['value'], errors='coerce')
        df['total']      = pd.to_numeric(df['total'], errors='coerce')
        df['match_date'] = pd.to_datetime(df['match_date'], utc=True, errors='coerce')

    print(f'\n{"="*55}')
    print(f'  Rows     : {len(df):,}')
    print(f'  Matches  : {df["match_id"].nunique() if not df.empty else 0}')
    print(f'  Players  : {df["player_id"].nunique() if not df.empty else 0}')
    print(f'  Failed   : {len(failed)}')
    print(f'{"="*55}')
    return df


# Uncomment one of these as needed:
# stats_df = reparse_from_cache(fixtures_df)                        # reparse raw JSON
# stats_df = pd.read_parquet(OUT_DIR / 'player_stats.parquet')      # reload saved parquet
print('✓ reparse_from_cache() defined')

✓ reparse_from_cache() defined


## Cell 7 — Inspect metric catalogue
Run this after scraping to see every metric that exists in the data.
The output drives which columns appear in the wide-format export.

In [141]:
occurrence = (
    stats_df
    .assign(player_type=stats_df['is_goalkeeper'].map({True: 'goalkeeper', False: 'outfield'}))
    .groupby(['player_type', 'category', 'metric_key', 'metric'])
    .agg(
        occurrences = ('value', 'count'),
        matches     = ('match_id', 'nunique'),
        players     = ('player_id', 'nunique'),
        pct_non_null= ('value', lambda x: f"{x.notna().mean()*100:.0f}%"),
        avg_value   = ('value', lambda x: round(x.mean(), 3)),
    )
    .reset_index()
    .sort_values(['player_type', 'category', 'occurrences'], ascending=[True, True, False])
    .reset_index(drop=True)
)

# Save for reference
occurrence.to_csv(OUT_DIR / 'metric_reference.csv', index=False)

header = f"{'metric_key':<45} {'category':<18} {'occur':>7} {'matches':>8} {'players':>8} {'non_null':>9} {'avg':>8}"
divider = '-' * 97

for ptype in ['outfield', 'goalkeeper']:
    subset = occurrence[occurrence['player_type'] == ptype]
    print(f'\n{"="*97}')
    print(f'{ptype.upper()} METRICS  ({subset["metric_key"].nunique()} unique)')
    print(f'{"="*97}')
    print(header)
    print(divider)
    for _, row in subset.iterrows():
        print(
            f"{row['metric_key']:<45} "
            f"{row['category']:<18} "
            f"{row['occurrences']:>7,} "
            f"{row['matches']:>8,} "
            f"{row['players']:>8,} "
            f"{row['pct_non_null']:>9} "
            f"{row['avg_value']:>8}"
        )

print(f'\n✓ Saved metric_reference.csv  ({len(occurrence)} total metrics)')


OUTFIELD METRICS  (48 unique)
metric_key                                    category             occur  matches  players  non_null      avg
-------------------------------------------------------------------------------------------------
dispossessed                                  attack               6,752      230      645      100%    0.585
touches                                       attack               6,752      230      645      100%   39.447
touches_opp_box                               attack               6,752      230      645      100%    1.472
passes_into_final_third                       attack               5,360      230      590      100%    4.289
long_balls_accurate                           attack               4,674      230      555      100%    1.693
dribbles_succeeded                            attack               3,388      230      489      100%    1.034
expected_goals_non_penalty                    attack               3,114      222      475      100% 

## Cell 8 — Build wide-format DataFrames
Fully dynamic — derives columns from whatever metrics exist in the data.
No hardcoded metric lists. Works for any league or season.

In [142]:
ID_COLS = [
    'match_id', 'round', 'match_date',
    'home_team', 'away_team',
    'player_id', 'player_name', 'team_id', 'team_name',
    'shirt_number', 'position_id', 'is_goalkeeper',
]


def build_wide(
    df          : pd.DataFrame,
    is_gk       : bool,
    min_matches : int   = 3,
    min_pct     : float = 0.0,
) -> pd.DataFrame:
    """
    Pivot long-format stats to wide (one row per player x match).

    Columns are derived automatically from whatever metrics exist
    in the data — no hardcoded lists. Works for any league/season.

    Fraction metrics (e.g. 23/26 passes) get a companion _total column
    so you can compute accuracy yourself: value / total.

    Args:
        df          : long-format stats_df
        is_gk       : True for goalkeepers, False for outfield
        min_matches : drop metrics that appear in fewer than this many matches
        min_pct     : drop metrics where fewer than this fraction of rows
                      have a non-null value (0.0 = keep everything)
    """
    subset = df[
        (df['is_goalkeeper'] == is_gk) &
        (~df['metric_key'].isin(EXCLUDE_FROM_WIDE)) &
        (df['metric_key'].notna()) &
        (df['metric_key'] != '')
    ].copy()

    # Dynamically decide which metric_keys to include
    metric_stats = (
        subset
        .groupby('metric_key')
        .agg(
            n_matches  = ('match_id', 'nunique'),
            pct_valid  = ('value', lambda x: x.notna().mean()),
            category   = ('category', 'first'),
            metric_label = ('metric', 'first'),
        )
        .reset_index()
    )

    # Apply filters
    included = metric_stats[
        (metric_stats['n_matches'] >= min_matches) &
        (metric_stats['pct_valid'] >= min_pct)
    ]

    # Order columns: top_stats first, then attack/passes/defense/duels/goalkeeping
    cat_order = ['top_stats', 'attack', 'passes', 'defense', 'duels', 'goalkeeping']
    included['cat_order'] = included['category'].map(
        {c: i for i, c in enumerate(cat_order)}
    ).fillna(99)
    included = included.sort_values(
        ['cat_order', 'n_matches'], ascending=[True, False]
    )
    valid_keys = included['metric_key'].tolist()

    # Filter to only included metrics
    subset = subset[subset['metric_key'].isin(valid_keys)]

    # Pivot values
    value_wide = subset.pivot_table(
        index=ID_COLS,
        columns='metric_key',
        values='value',
        aggfunc='first',
    ).reset_index()
    value_wide.columns.name = None

    # Pivot totals for fraction metrics
    frac = subset[subset['stat_type'] == 'fractionWithPercentage']
    if not frac.empty:
        total_wide = frac.pivot_table(
            index=ID_COLS,
            columns='metric_key',
            values='total',
            aggfunc='first',
        ).reset_index()
        total_wide.columns.name = None
        rename_map = {c: f'{c}_total' for c in total_wide.columns if c not in ID_COLS}
        total_wide = total_wide.rename(columns=rename_map)
        wide = value_wide.merge(
            total_wide[ID_COLS + list(rename_map.values())],
            on=ID_COLS, how='left',
        )
    else:
        wide = value_wide

    # Final column order: ID cols → metric values (ordered) → totals
    val_cols   = [c for c in valid_keys if c in wide.columns]
    total_cols = [c for c in wide.columns if c.endswith('_total')]
    other_cols = [c for c in wide.columns if c not in ID_COLS + val_cols + total_cols]
    ordered    = ID_COLS + val_cols + total_cols + other_cols
    ordered    = [c for c in ordered if c in wide.columns]

    wide = wide[ordered].sort_values(['round', 'match_id', 'player_id']).reset_index(drop=True)

    print(f'  {"Goalkeeper" if is_gk else "Outfield":12} → {len(wide):>6,} rows  x  {wide.shape[1]:>3} cols  ({len(val_cols)} metrics)')
    return wide


print('Building wide-format tables...')
print()
outfield_df = build_wide(stats_df, is_gk=False)
gk_df       = build_wide(stats_df, is_gk=True)

print()
print('Outfield metric columns:')
print([c for c in outfield_df.columns if c not in ID_COLS and not c.endswith('_total')])
print()
print('Goalkeeper metric columns:')
print([c for c in gk_df.columns if c not in ID_COLS and not c.endswith('_total')])

Building wide-format tables...

  Outfield     →  6,750 rows  x   67 cols  (48 metrics)
  Goalkeeper   →    460 rows  x   55 cols  (38 metrics)

Outfield metric columns:
['ShotsOffTarget', 'ShotsOnTarget', 'accurate_passes', 'assists', 'blocked_shots', 'chances_created', 'defensive_actions', 'goals', 'rating_title', 'shot_accuracy', 'minutes_played', 'expected_assists', 'expected_goals', 'expected_goals_on_target_variant', 'xg_and_xa', 'big_chance_created_team_title', 'conceded_penalties', 'penalties_won', 'errors_led_to_goal', 'owngoal', 'missed_penalty', 'accurate_crosses', 'corners', 'dispossessed', 'dribbles_succeeded', 'long_balls_accurate', 'passes_into_final_third', 'touches', 'touches_opp_box', 'expected_goals_non_penalty', 'Offsides', 'big_chance_missed_title', 'shots_woodwork', 'clearances', 'dribbled_past', 'headed_clearance', 'interceptions', 'matchstats.headers.tackles', 'recoveries', 'shot_blocks', 'last_man_tackle', 'clearance_off_the_line', 'aerials_won', 'duel_lost', '

## Cell 9 — Save wide-format CSVs + Parquet

In [143]:
outfield_df.to_parquet(OUT_DIR / 'outfield_players.parquet', index=False, engine='pyarrow')
outfield_df.to_csv(OUT_DIR / 'outfield_players.csv', index=False)

gk_df.to_parquet(OUT_DIR / 'goalkeepers.parquet', index=False, engine='pyarrow')
gk_df.to_csv(OUT_DIR / 'goalkeepers.csv', index=False)

print('File                        Parquet       CSV')
print('-' * 52)
for name in ['outfield_players', 'goalkeepers', 'player_stats', 'fixtures']:
    pq  = OUT_DIR / f'{name}.parquet'
    csv = OUT_DIR / f'{name}.csv'
    pq_mb  = pq.stat().st_size  / 1e6 if pq.exists()  else 0
    csv_mb = csv.stat().st_size / 1e6 if csv.exists() else 0
    print(f'{name:<28} {pq_mb:>5.1f} MB   {csv_mb:>5.1f} MB')

File                        Parquet       CSV
----------------------------------------------------
outfield_players               0.3 MB     1.8 MB
goalkeepers                    0.1 MB     0.1 MB
player_stats                   0.6 MB    35.8 MB
fixtures                       0.0 MB     0.0 MB


## Cell 10 — Sanity check queries

In [144]:
# Reload from parquet to confirm files are clean
outfield_check = pd.read_parquet(OUT_DIR / 'outfield_players.parquet')
gk_check       = pd.read_parquet(OUT_DIR / 'goalkeepers.parquet')

assert len(outfield_check) == len(outfield_df), '⚠ Outfield row count mismatch'
assert len(gk_check)       == len(gk_df),       '⚠ Goalkeeper row count mismatch'
print('✓ Parquet files match in-memory DataFrames')
print(f'  Outfield : {len(outfield_check):,} rows x {outfield_check.shape[1]} cols')
print(f'  GK       : {len(gk_check):,} rows x {gk_check.shape[1]} cols')
print()

# Top rated outfield players (min 5 appearances)
if 'rating_title' in outfield_check.columns:
    top_rated = (
        outfield_check
        .dropna(subset=['rating_title'])
        .groupby(['player_id', 'player_name', 'team_name'])
        .agg(appearances=('match_id', 'count'), avg_rating=('rating_title', 'mean'))
        .query('appearances >= 5')
        .sort_values('avg_rating', ascending=False)
        .head(10)
        .reset_index()
    )
    print('── Top 10 rated outfield players (min 5 apps) ──')
    print(top_rated[['player_name', 'team_name', 'appearances', 'avg_rating']].to_string(index=False))
    print()

# Goals leaderboard
if 'goals' in outfield_check.columns:
    goals = (
        outfield_check
        .groupby(['player_id', 'player_name', 'team_name'])['goals']
        .sum().sort_values(ascending=False)
        .head(10).reset_index()
    )
    print('── Top 10 scorers ──')
    print(goals[['player_name', 'team_name', 'goals']].to_string(index=False))
    print()

# Goalkeeper saves leaderboard
if 'saves' in gk_check.columns:
    saves = (
        gk_check
        .groupby(['player_id', 'player_name', 'team_name'])['saves']
        .sum().sort_values(ascending=False)
        .head(10).reset_index()
    )
    print('── Top 10 GK by saves ──')
    print(saves[['player_name', 'team_name', 'saves']].to_string(index=False))

✓ Parquet files match in-memory DataFrames
  Outfield : 6,750 rows x 67 cols
  GK       : 460 rows x 55 cols

── Top 10 rated outfield players (min 5 apps) ──
        player_name   team_name  appearances  avg_rating
       João Cancelo   Barcelona            6    8.256667
      Kylian Mbappé Real Madrid           10    8.134000
       Lamine Yamal   Barcelona           20    8.077000
    Vinícius Júnior Real Madrid            8    7.845000
    Alvaro Carreras Real Madrid           12    7.785000
    Frenkie de Jong   Barcelona            8    7.760000
Aurélien Tchouaméni Real Madrid           16    7.735000
        Eric Garcia   Barcelona            8    7.675000
         Javi Galán     Osasuna           16    7.650000
  Federico Valverde Real Madrid           20    7.635000

── Top 10 scorers ──
       player_name          team_name  goals
      Lamine Yamal          Barcelona   16.0
      Vedat Muriqi           Mallorca   16.0
   Mikel Oyarzabal      Real Sociedad   16.0
      Ante B